# Feature Engineering Notebook for Credit Default Data




In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

DATA_PATH = Path('data/credit-default-data.csv')
TARGET = 'Credit Default'


## Load Training Data

Only the historical labeled training dataset is used here. We keep the target column untouched and prepare all feature transformations around it.

In [2]:
df = pd.read_csv(DATA_PATH)

print(f'Training data shape: {df.shape}')
display(df.head())

schema_overview = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'nunique': df.nunique(dropna=False)
})
display(schema_overview)


Training data shape: (7500, 18)


,Id,Home Ownership,Annual Income,Years in current job,Tax Liens,Number of Open Accounts,Years of Credit History,Maximum Open Credit,Number of Credit Problems,Months since last delinquent,Bankruptcies,Purpose,Term,Current Loan Amount,Current Credit Balance,Monthly Debt,Credit Score,Credit Default
0,0,Own Home,"482,087.000",NaN,0.000,11.000,26.300,"685,960.000",1.000,NaN,1.000,debt consolidation,Short Term,"99,999,999.000","47,386.000","7,914.000",749.000,0
1,1,Own Home,"1,025,487.000",10+ years,0.000,15.000,15.300,"1,181,730.000",0.000,NaN,0.000,debt consolidation,Long Term,"264,968.000","394,972.000","18,373.000",737.000,1
2,2,Home Mortgage,"751,412.000",8 years,0.000,11.000,35.000,"1,182,434.000",0.000,NaN,0.000,debt consolidation,Short Term,"99,999,999.000","308,389.000","13,651.000",742.000,0
3,3,Own Home,"805,068.000",6 years,0.000,8.000,22.500,"147,400.000",1.000,NaN,1.000,debt consolidation,Short Term,"121,396.000","95,855.000","11,338.000",694.000,0
4,4,Rent,"776,264.000",8 years,0.000,13.000,13.600,"385,836.000",1.000,NaN,0.000,debt consolidation,Short Term,"125,840.000","93,309.000","7,180.000",719.000,0


,dtype,missing_count,missing_pct,nunique
Id,int64,0,0.000,7500
Home Ownership,object,0,0.000,4
Annual Income,float64,1557,20.760,5479
Years in current job,object,371,4.950,12
Tax Liens,float64,0,0.000,8
Number of Open Accounts,float64,0,0.000,39
Years of Credit History,float64,0,0.000,408
Maximum Open Credit,float64,0,0.000,6963
Number of Credit Problems,float64,0,0.000,8
Months since last delinquent,float64,4081,54.410,90


## Transformation Rules

- `Current Loan Amount == 99999999` is treated as a process placeholder, so we convert it into a separate flag and then impute the original amount column.
- `Credit Score > 850` is treated as a scaling issue, so those values are corrected by dividing by `10` before imputation.
- Missingness flags are kept for the most informative columns because missing data itself may carry operational signal.
- This notebook prepares features only; feature importance belongs in the later modeling notebook after train/test splitting.

In [3]:
def job_tenure_to_years(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if value == '< 1 year':
        return 0.5
    if value == '10+ years':
        return 10.0
    match = re.search(r'(\d+)', value)
    return float(match.group(1)) if match else np.nan


## Create Flags and Clean Known Data Issues

We create missingness and placeholder flags before imputation so the final dataset retains operational information instead of silently hiding it.

In [4]:
working_df = df.copy()

working_df['income_missing_flag'] = working_df['Annual Income'].isna().astype(int)
working_df['credit_score_missing_flag'] = working_df['Credit Score'].isna().astype(int)
working_df['job_tenure_missing_flag'] = working_df['Years in current job'].isna().astype(int)
working_df['delinquency_missing_flag'] = working_df['Months since last delinquent'].isna().astype(int)

working_df['loan_amount_placeholder_flag'] = working_df['Current Loan Amount'].eq(99999999).astype(int)
working_df['Home Ownership'] = working_df['Home Ownership'].replace({'Have Mortgage': 'Home Mortgage'})
working_df['Current Loan Amount'] = working_df['Current Loan Amount'].mask(working_df['loan_amount_placeholder_flag'].eq(1))
working_df['Credit Score'] = working_df['Credit Score'].where(working_df['Credit Score'] <= 850, working_df['Credit Score'] / 10)
working_df['job_tenure_years'] = working_df['Years in current job'].apply(job_tenure_to_years)

cleaning_summary = pd.DataFrame({
    'metric': [
        'Rows flagged as loan amount placeholder',
        'Rows originally missing Annual Income',
        'Rows originally missing Credit Score',
        'Rows originally missing Years in current job',
        'Rows originally missing Months since last delinquent',
        'Rows with Credit Score scaled down from > 850'
    ],
    'value': [
        int(working_df['loan_amount_placeholder_flag'].sum()),
        int(working_df['income_missing_flag'].sum()),
        int(working_df['credit_score_missing_flag'].sum()),
        int(working_df['job_tenure_missing_flag'].sum()),
        int(working_df['delinquency_missing_flag'].sum()),
        int((df['Credit Score'] > 850).sum())
    ]
})

display(cleaning_summary)
display(working_df[['Home Ownership', 'Current Loan Amount', 'loan_amount_placeholder_flag', 'Credit Score', 'job_tenure_years']].head())


,metric,value
0,Rows flagged as loan amount placeholder,870
1,Rows originally missing Annual Income,1557
2,Rows originally missing Credit Score,1557
3,Rows originally missing Years in current job,371
4,Rows originally missing Months since last deli...,4081
5,Rows with Credit Score scaled down from > 850,400


,Home Ownership,Current Loan Amount,loan_amount_placeholder_flag,Credit Score,job_tenure_years
0,Own Home,NaN,1,749.000,NaN
1,Own Home,"264,968.000",0,737.000,10.000
2,Home Mortgage,NaN,1,742.000,8.000
3,Own Home,"121,396.000",0,694.000,6.000
4,Rent,"125,840.000",0,719.000,8.000


## Median Imputation for Source Columns

Median imputation is used for the planned source columns because it is simple, robust to skew, and consistent with a later train/test modeling workflow. The missingness flags created above retain information about whether the value was originally absent.

In [5]:
imputation_columns = [
    'Annual Income',
    'Credit Score',
    'Current Loan Amount',
    'job_tenure_years',
    'Months since last delinquent',
    'Bankruptcies'
]

imputation_summary = []
for column in imputation_columns:
    median_value = float(working_df[column].median())
    missing_before = int(working_df[column].isna().sum())
    working_df[column] = working_df[column].fillna(median_value)
    missing_after = int(working_df[column].isna().sum())
    imputation_summary.append({
        'column': column,
        'median_used': median_value,
        'missing_before': missing_before,
        'missing_after': missing_after
    })

working_df = working_df.drop(columns=['Id', 'Years in current job'])

display(pd.DataFrame(imputation_summary))


,column,median_used,missing_before,missing_after
0,Annual Income,"1,168,386.000",1557,0
1,Credit Score,729.000,1557,0
2,Current Loan Amount,"265,826.000",870,0
3,job_tenure_years,6.000,371,0
4,Months since last delinquent,32.000,4081,0
5,Bankruptcies,0.000,14,0


## Engineer Final Numeric Features

These engineered variables translate repayment burden, leverage, and exposure size into ratios that are more useful for modeling than the raw columns alone.

In [6]:
working_df['monthly_dti'] = working_df['Monthly Debt'] / (working_df['Annual Income'] / 12)
working_df['credit_utilization_proxy'] = working_df['Current Credit Balance'] / working_df['Maximum Open Credit'].replace(0, np.nan)
working_df['loan_to_income_ratio'] = working_df['Current Loan Amount'] / working_df['Annual Income']

utilization_missing_before = int(working_df['credit_utilization_proxy'].isna().sum())
utilization_median = float(working_df['credit_utilization_proxy'].median())
working_df['credit_utilization_proxy'] = working_df['credit_utilization_proxy'].fillna(utilization_median)
utilization_missing_after = int(working_df['credit_utilization_proxy'].isna().sum())

engineered_summary = pd.DataFrame({
    'feature': ['monthly_dti', 'credit_utilization_proxy', 'loan_to_income_ratio'],
    'missing_count': [
        int(working_df['monthly_dti'].isna().sum()),
        int(working_df['credit_utilization_proxy'].isna().sum()),
        int(working_df['loan_to_income_ratio'].isna().sum())
    ],
    'mean': [
        float(working_df['monthly_dti'].mean()),
        float(working_df['credit_utilization_proxy'].mean()),
        float(working_df['loan_to_income_ratio'].mean())
    ]
})

display(engineered_summary)
print('credit_utilization_proxy missing before imputation:', utilization_missing_before)
print('credit_utilization_proxy median used:', utilization_median)
print('credit_utilization_proxy missing after imputation:', utilization_missing_after)


,feature,missing_count,mean
0,monthly_dti,0,0.175
1,credit_utilization_proxy,0,0.477
2,loan_to_income_ratio,0,0.254


credit_utilization_proxy missing before imputation: 65
credit_utilization_proxy median used: 0.4879531188996006
credit_utilization_proxy missing after imputation: 0


## Assemble the Prepared Training Dataset

The readable prepared dataframe keeps cleaned source columns, engineered features, flags, and the target together before categorical encoding.

In [7]:
base_numeric_cols = [
    'Annual Income',
    'Tax Liens',
    'Number of Open Accounts',
    'Years of Credit History',
    'Maximum Open Credit',
    'Number of Credit Problems',
    'Months since last delinquent',
    'Bankruptcies',
    'Current Loan Amount',
    'Current Credit Balance',
    'Monthly Debt',
    'Credit Score',
    'job_tenure_years'
]

engineered_numeric_cols = [
    'monthly_dti',
    'credit_utilization_proxy',
    'loan_to_income_ratio'
]

flag_cols = [
    'loan_amount_placeholder_flag',
    'income_missing_flag',
    'credit_score_missing_flag',
    'job_tenure_missing_flag',
    'delinquency_missing_flag'
]

categorical_cols = ['Home Ownership', 'Purpose', 'Term']

prepared_training_cols = base_numeric_cols + engineered_numeric_cols + flag_cols + categorical_cols + [TARGET]
prepared_training_df = working_df[prepared_training_cols].copy()

feature_group_summary = pd.DataFrame({
    'group': ['base_numeric_cols', 'engineered_numeric_cols', 'flag_cols', 'categorical_cols'],
    'count': [len(base_numeric_cols), len(engineered_numeric_cols), len(flag_cols), len(categorical_cols)]
})

display(feature_group_summary)
display(prepared_training_df.head())


,group,count
0,base_numeric_cols,13
1,engineered_numeric_cols,3
2,flag_cols,5
3,categorical_cols,3


,Annual Income,Tax Liens,Number of Open Accounts,Years of Credit History,Maximum Open Credit,Number of Credit Problems,Months since last delinquent,Bankruptcies,Current Loan Amount,Current Credit Balance,Monthly Debt,Credit Score,job_tenure_years,monthly_dti,credit_utilization_proxy,loan_to_income_ratio,loan_amount_placeholder_flag,income_missing_flag,credit_score_missing_flag,job_tenure_missing_flag,delinquency_missing_flag,Home Ownership,Purpose,Term,Credit Default
0,"482,087.000",0.000,11.000,26.300,"685,960.000",1.000,32.000,1.000,"265,826.000","47,386.000","7,914.000",749.000,6.000,0.197,0.069,0.551,1,0,0,1,1,Own Home,debt consolidation,Short Term,0
1,"1,025,487.000",0.000,15.000,15.300,"1,181,730.000",0.000,32.000,0.000,"264,968.000","394,972.000","18,373.000",737.000,10.000,0.215,0.334,0.258,0,0,0,0,1,Own Home,debt consolidation,Long Term,1
2,"751,412.000",0.000,11.000,35.000,"1,182,434.000",0.000,32.000,0.000,"265,826.000","308,389.000","13,651.000",742.000,8.000,0.218,0.261,0.354,1,0,0,0,1,Home Mortgage,debt consolidation,Short Term,0
3,"805,068.000",0.000,8.000,22.500,"147,400.000",1.000,32.000,1.000,"121,396.000","95,855.000","11,338.000",694.000,6.000,0.169,0.650,0.151,0,0,0,0,1,Own Home,debt consolidation,Short Term,0
4,"776,264.000",0.000,13.000,13.600,"385,836.000",1.000,32.000,0.000,"125,840.000","93,309.000","7,180.000",719.000,8.000,0.111,0.242,0.162,0,0,0,0,1,Rent,debt consolidation,Short Term,0


## Create the Model-Ready Matrix

We one-hot encode the categorical fields and keep the numeric features as numeric. This produces a deterministic model-ready matrix with no missing values and no object dtypes.

In [8]:
feature_cols_pre_encoding = base_numeric_cols + engineered_numeric_cols + flag_cols + categorical_cols

X_model = pd.get_dummies(
    prepared_training_df[feature_cols_pre_encoding],
    columns=categorical_cols,
    drop_first=False,
    dtype=int
).sort_index(axis=1)

y = prepared_training_df[TARGET].copy()
model_training_df = X_model.copy()
model_training_df[TARGET] = y

shape_summary = pd.DataFrame({
    'object_name': ['prepared_training_df', 'X_model', 'y', 'model_training_df'],
    'rows': [prepared_training_df.shape[0], X_model.shape[0], y.shape[0], model_training_df.shape[0]],
    'columns': [prepared_training_df.shape[1], X_model.shape[1], 1, model_training_df.shape[1]]
})

display(shape_summary)
display(model_training_df.head())


,object_name,rows,columns
0,prepared_training_df,7500,25
1,X_model,7500,41
2,y,7500,1
3,model_training_df,7500,42


,Annual Income,Bankruptcies,Credit Score,Current Credit Balance,Current Loan Amount,Home Ownership_Home Mortgage,Home Ownership_Own Home,Home Ownership_Rent,Maximum Open Credit,Monthly Debt,Months since last delinquent,Number of Credit Problems,Number of Open Accounts,Purpose_business loan,Purpose_buy a car,Purpose_buy house,Purpose_debt consolidation,Purpose_educational expenses,Purpose_home improvements,Purpose_major purchase,Purpose_medical bills,Purpose_moving,Purpose_other,Purpose_renewable energy,Purpose_small business,Purpose_take a trip,Purpose_vacation,Purpose_wedding,Tax Liens,Term_Long Term,Term_Short Term,Years of Credit History,credit_score_missing_flag,credit_utilization_proxy,delinquency_missing_flag,income_missing_flag,job_tenure_missing_flag,job_tenure_years,loan_amount_placeholder_flag,loan_to_income_ratio,monthly_dti,Credit Default
0,"482,087.000",1.000,749.000,"47,386.000","265,826.000",0,1,0,"685,960.000","7,914.000",32.000,1.000,11.000,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.000,0,1,26.300,0,0.069,1,0,1,6.000,1,0.551,0.197,0
1,"1,025,487.000",0.000,737.000,"394,972.000","264,968.000",0,1,0,"1,181,730.000","18,373.000",32.000,0.000,15.000,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.000,1,0,15.300,0,0.334,1,0,0,10.000,0,0.258,0.215,1
2,"751,412.000",0.000,742.000,"308,389.000","265,826.000",1,0,0,"1,182,434.000","13,651.000",32.000,0.000,11.000,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.000,0,1,35.000,0,0.261,1,0,0,8.000,1,0.354,0.218,0
3,"805,068.000",1.000,694.000,"95,855.000","121,396.000",0,1,0,"147,400.000","11,338.000",32.000,1.000,8.000,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.000,0,1,22.500,0,0.650,1,0,0,6.000,0,0.151,0.169,0
4,"776,264.000",0.000,719.000,"93,309.000","125,840.000",0,0,1,"385,836.000","7,180.000",32.000,1.000,13.000,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.000,0,1,13.600,0,0.242,1,0,0,8.000,0,0.162,0.111,0
